# Customer Discovery Agent: Ollama test

Runs the funnel with real local models (`nomic-embed-text` + `llama3.1:8b`) on Colab, so nothing is downloaded to your machine.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then Runtime > Run all. When cell 2 asks, upload `discovery_project.zip`.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Upload the project zip and unpack it
import os, zipfile
from google.colab import files

up = files.upload()
name = next(iter(up))
os.makedirs("/content/project", exist_ok=True)
zipfile.ZipFile(name).extractall("/content/project")
os.chdir("/content/project")
!ls

In [ ]:
# 3. Install Ollama and start the server (about 1-2 min)
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

import subprocess, time, urllib.request
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434", timeout=2)
        print("ollama is up")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama server did not start")

In [ ]:
# 4. Pull the two models (about 5 GB on Colab's connection, a few minutes)
!ollama pull nomic-embed-text
!ollama pull llama3.1:8b
!ollama list

In [ ]:
# 5. Sanity check: unit tests and synthetic data
!python -m unittest discover -s tests 2>&1 | tail -4
!python -m discovery.synthetic --days 3 --per-day 1000 --out data/synthetic.csv

In [ ]:
# 6. Stage-2 threshold sweep (embeddings only, no LLM calls, fast)
# For messages that pass stage 1, show how many real leads / noise messages survive each threshold.
import sys
sys.path.insert(0, "/content/project")
from discovery.backends import OllamaEmbedder
from discovery.stages import PreFilterStage, EmbeddingStage
from discovery.models import Candidate
from discovery.sources import CsvSource
from discovery.settings import DEFAULT_LIBRARY

LIMIT = 1500
emb = OllamaEmbedder("http://localhost:11434", "nomic-embed-text")
pre = PreFilterStage(12)
st = EmbeddingStage(emb, DEFAULT_LIBRARY, -1.0)

rows, total_leads = [], 0
for m in CsvSource("data/synthetic.csv", LIMIT).messages():
    total_leads += int(m.label == 1)
    c = Candidate(m)
    if pre.process(c):
        st.process(c)
        rows.append((c.similarity, m.label))

pass1_leads = sum(1 for _, l in rows if l == 1)
print(f"{LIMIT} msgs, {total_leads} real leads; {len(rows)} passed stage 1 ({pass1_leads} real leads)")
print(f"{'thr':>5} {'kept':>6} {'leads kept':>11} {'noise kept':>11} {'lead recall vs all':>19}")
for t in [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85]:
    kept = [(s, l) for s, l in rows if s >= t]
    lk = sum(1 for _, l in kept if l == 1)
    print(f"{t:>5} {len(kept):>6} {lk:>11} {len(kept)-lk:>11} {lk/total_leads:>19.2f}")

In [ ]:
# 7. Full funnel with real models at several thresholds, local only (no Claude).
# Embedding and LLM results are cached per message, so the extra thresholds cost almost nothing.
from discovery.backends import OllamaEmbedder, OllamaScorer
from discovery.factory import build_pipeline
from discovery.settings import Settings
from discovery.report import render

def memo(cls, meth):
    orig, cache = getattr(cls, meth), {}
    def wrapped(self, text):
        if text not in cache:
            cache[text] = orig(self, text)
        return cache[text]
    setattr(cls, meth, wrapped)

memo(OllamaEmbedder, "embed")
memo(OllamaScorer, "score")

THRESHOLDS = [0.50, 0.55, 0.60]
out, runs = [], {}
for T in THRESHOLDS:
    pipe = build_pipeline(Settings(backend="ollama", embed_threshold=T, db_path=""), use_cloud=False)
    s = pipe.run(CsvSource("data/synthetic.csv", 1500), f"T={T}")
    runs[T] = (s, pipe)

out.append(f"{'thr':>5} {'leads':>6} {'TP':>4} {'FP':>4} {'FN':>4} {'prec':>6} {'recall':>7} {'s2 in':>6} {'s3 in':>6} {'s3 lost leads':>14} {'s3 noise dropped':>17}")
for T, (s, pipe) in runs.items():
    prec = s.tp / (s.tp + s.fp) if (s.tp + s.fp) else 0.0
    rec = s.tp / (s.tp + s.fn) if (s.tp + s.fn) else 0.0
    s3 = s.stages["3_local_llm"]
    out.append(f"{T:>5} {len(s.leads):>6} {s.tp:>4} {s.fp:>4} {s.fn:>4} {prec:>6.2f} {rec:>7.2f} {s.stages['2_embedding']['out']:>6} {s3['in']:>6} {s3['leads_lost']:>14} {s3['noise_dropped']:>17}")

for T, (s, pipe) in runs.items():
    fps = [c for c in s.leads if c.message.label == 0]
    out.append(f"\n--- false positives at T={T}: {len(fps)} ---")
    for c in fps[:10]:
        out.append(f"  {c.intent_score:.2f} conf {c.confidence:.2f}  {c.message.text}")

s, pipe = runs[0.55]
out.append("\n=== full report at T=0.55 ===")
out.append(render(s, pipe.tracker, 10))

text = "\n".join(out)
print(text)
open("report_sweep.txt", "w").write(text)

In [ ]:
# 8. Offline baseline on the same 1500 messages, for comparison
!python -m discovery --csv data/synthetic.csv --db "" --no-cloud --limit 1500 2>&1 | tee report_offline.txt

In [ ]:
# 9. Download the reports (or copy the output of cells 6, 7 and 8 back into the chat)
from google.colab import files
files.download("report_sweep.txt")
files.download("report_offline.txt")